# Laya × TrainGame: fine-tuning su Kaggle o Colab

Mette a punto `convaiinnovations/laya-multilingual` (commit `e4e9ddf2`) sulle decisioni degli NPC
esportate da `export_dataset`, con `finetune.py` (ricetta RLCD del notebook ufficiale di Laya).
Istruzioni complete: `tools/laya-finetune/README.md`.

**Kaggle:** Settings → Accelerator **GPU T4 x2** (lo script ne usa una) · Internet **On** ·
Add Input → il tuo dataset con `train.jsonl`, `val.jsonl`, `test.jsonl`, `finetune.py`, `requirements.txt`.

**Colab:** Runtime → Change runtime type → **T4 GPU**; carica la cartella (o lo zip) nella cella 2.

In [ ]:
!nvidia-smi

In [ ]:
# Trova la cartella con il dataset e lo script (Kaggle: /kaggle/input/…, Colab: /content/…).
import glob, os
hits = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
              + glob.glob("/content/**/train.jsonl", recursive=True))
assert hits, "Nessun train.jsonl: aggiungi il dataset (Kaggle) o carica/scompatta la cartella (Colab)"
DATA = os.path.dirname(hits[0])
KIT = DATA if os.path.exists(os.path.join(DATA, "finetune.py")) else "."
OUT = "/kaggle/working/laya-traingame-ft" if os.path.isdir("/kaggle/working") else "/content/laya-traingame-ft"
print("DATA =", DATA, "· KIT =", KIT, "· OUT =", OUT)

In [ ]:
!pip install -q -r {KIT}/requirements.txt

In [ ]:
# Controllo del dataset (senza GPU): righe per split, etichette, posizioni.
!python {KIT}/finetune.py --data {DATA} --check-data

In [ ]:
# Training + calibrazione + valutazione + salvataggio (~1 h su una T4 con i default; vedi README).
# Più veloce ma meno potente: aggiungi --mode head. Poca memoria: --micro-batch 4 --grad-accum 8 --grad-ckpt
!python {KIT}/finetune.py --data {DATA} --out {OUT}

Fatto: scarica `laya-traingame-ft.zip` (Kaggle: pannello **Output**; Colab: pannello **File**,
o copialo su Drive), scompattalo e usalo nel gioco con `LAYA_MODEL_DIR` (vedi README, passo 4).